#### Importing Libraries

In [8]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_squared_error,
    r2_score,
)

import statsmodels.api as sm

#### Importing Zillow Home Value Index (ZHVI) dataset

In [9]:
zillow_hv_index = pd.read_csv("../data/raw/Zillow/zillow_home_value_index.csv")

#### Initial Overview

In [10]:
metadata_df = zillow_hv_index.iloc[:, :5]
monthly_values_df = zillow_hv_index.iloc[:, 5:]

print(
    f"""=== Dataset Dimensions ===
Rows: {zillow_hv_index.shape[0]:,}
Columns: {zillow_hv_index.shape[1]:,}

=== Dataset Structure ===
Metadata Columns: {metadata_df.shape[1]}
Monthly Value Columns: {monthly_values_df.shape[1]}
"""
)

print("=== Monthly Values Overview ===")
monthly_values_df.info()

print("\n=== Metadata Overview ===")
metadata_df.info()

=== Dataset Dimensions ===
Rows: 895
Columns: 322

=== Dataset Structure ===
Metadata Columns: 5
Monthly Value Columns: 317

=== Monthly Values Overview ===
<class 'pandas.DataFrame'>
RangeIndex: 895 entries, 0 to 894
Columns: 317 entries, 2000-01-31 to 2026-05-31
dtypes: float64(317)
memory usage: 2.2 MB

=== Metadata Overview ===
<class 'pandas.DataFrame'>
RangeIndex: 895 entries, 0 to 894
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   RegionID    895 non-null    int64
 1   SizeRank    895 non-null    int64
 2   RegionName  895 non-null    str  
 3   RegionType  895 non-null    str  
 4   StateName   894 non-null    str  
dtypes: int64(2), str(3)
memory usage: 35.1 KB


#### Data Cleaning

In [11]:
# Create separate variable for first 5 columns, because after that the columns are monthly updates to the housing index per region
meta_data = zillow_hv_index.columns[:5]

# Check if there's any instance in the meta data where the row is entirely null, if so we'll need to remove, in this case there's none
zillow_hv_index[ zillow_hv_index[meta_data].isna().all(axis=1) ]

,RegionID,SizeRank,RegionName,RegionType,StateName,2000-01-31,2000-02-29,2000-03-31,2000-04-30,2000-05-31,...,2025-08-31,2025-09-30,2025-10-31,2025-11-30,2025-12-31,2026-01-31,2026-02-28,2026-03-31,2026-04-30,2026-05-31


In [12]:
# Create separate variable for all columns after the first 5, because after that the columns are monthly updates to the housing index per region
time_series = zillow_hv_index.columns[5:]

# Check if there's any instance in the time series where the row is entirely null, if so we'll need to remove, in this case there's none
zillow_hv_index[ zillow_hv_index[time_series].isna().all(axis=1) ]

,RegionID,SizeRank,RegionName,RegionType,StateName,2000-01-31,2000-02-29,2000-03-31,2000-04-30,2000-05-31,...,2025-08-31,2025-09-30,2025-10-31,2025-11-30,2025-12-31,2026-01-31,2026-02-28,2026-03-31,2026-04-30,2026-05-31


In [18]:
# Remove the national aggregate row so the dataset contains only MSA-level observations
zillow_hv_index = zillow_hv_index[
    zillow_hv_index["RegionType"] == "msa"]

In [19]:
# Checking duplicates, none in this case
zillow_hv_index.duplicated().sum()

np.int64(0)